In [80]:
import json
import pandas as pd
import numpy as np

In [7]:
with open("data/processed.json") as file:
    data = json.load(file)

In [35]:
df = pd.DataFrame(data)
df.head()

,CaseNo,Judgement,ArgumentsOfPetitioner,ArgumentsOfResponder,Facts,Principle,Statute,CaseCited,Reasoning,Decision,NoOfJudges,Decision_Status
0,Civil Appeal No. 6006 Of 2001 (With C.A. No. 3...,Dharmadhikari J.\t\tThese two cross appeals ha...,32. It needs to be mentioned at this stage tha...,44. The alternative argument advanced on behal...,1.These two cross appeals have been preferred ...,1.These two cross appeals have been preferred ...,"Specific Relief Act, 1963 Section 20&nbsp;",Lala Durga Prasad vs. Lala Deep Chand. (1954 S...,After hearing the argument at length advanced ...,&nbsp;70. On duly appreciating of the evidence...,2,2
1,Civil Appeal No. 4674 Of 1998 (With C.A. No. 4...,"Srikrishna, J.1.&nbsp;Leave granted in the spe...",&nbsp;Learned counsel for the appellants conte...,&nbsp;Learned counsel for the State Government...,The appellants manufacture gutka within the st...,&nbsp;The common issue raised for consideratio...,"Prevention of Food Adulteration Act, 1954 - Se...",Gandhi Irwin Salt Manufacturers Association vs...,There appears to be merit in the contentions o...,"In the result, we allow the appeals and the wr...",2,1
2,Civil Appeal No. 8452 Of 2003 (With C.A. No. 8...,"Arijit Pasayat, J.:-1. Leave granted.2. Appell...","&nbsp;On behalf of the appellant-workmen, rely...","7. Per contra, on behalf of the respondent Man...",2. Appellants contend that the view which was ...,2. Appellants contend that the view which was ...,Maharashtra Recognition of Trade Unions &amp; ...,1.Steel Authority of India Ltd. and Ors. Vs. N...,CIPLA&#39;s case (supra) where detailed analys...,These appeals are without merit and deserve di...,2,0
3,Civil Appeal No. 6790 Of 2003,"H.K. Sema, J.1. This appeal is directed agains...","24. Mr. Rohtagi, learned senior counsel, has d...","12. Dr. Singhvi, learned senior counsel appear...",3. The 1st appellant is the wife of the 5th re...,This appeal is directed against the judgment o...,"Code of Civil Procedure, 1908 -","Vidhyadhar vs. Manikrao and Another, (1999) 3 ...",15. Having regard to the directions in the ord...,"Accordingly, the appeal fails and is dismissed...",2,2
4,Civil Appeal No. 439 Of 1997 (With C.A. No. 84...,"S.N. Variava, J.:-1. Leave granted.2. Both the...",13. Written submissions on behalf of the Appel...,In this behalf of the relevant portions of the...,"On 28th of August, 1993, the Appellants appear...",\tWhether a co-operative society registered un...,"Banking Regulation Act, 1949 - Section 22&nbsp...",\tR.C. Cooper vs. Union of India reported in (...,17. We are unable to accept these submissions ...,In view of what we have held we direct the RBI...,2,1


In [13]:
from tokenizers import BertWordPieceTokenizer, ByteLevelBPETokenizer, SentencePieceBPETokenizer

In [16]:
import wget
urls = ["https://s3.amazonaws.com/models.huggingface.co/bert/bert-large-uncased-vocab.txt",
"https://s3.amazonaws.com/models.huggingface.co/bert/bert-large-cased-vocab.txt"]

for i in urls:
    filename = wget.download(i)
    print(filename, end=" ")

bert-large-uncased-vocab (1).txt bert-large-cased-vocab (1).txt 

In [17]:
# BERT vocabularies
bertLargeCased = "bert-large-cased-vocab.txt"
bertLargeUncased = "bert-large-uncased-vocab.txt"

wordPiece_cased = BertWordPieceTokenizer(bertLargeCased)
wordPiece_cased_encoder = wordPiece_cased.encode(words)

wordPiece_uncased = BertWordPieceTokenizer(bertLargeUncased)
wordPiece_uncased_encoder = wordPiece_uncased.encode(words)

### Checking tokenizers

In [23]:
print(len(wordPiece_cased_encoder.ids))
print(wordPiece_cased_encoder.tokens[100:150])
print(wordPiece_cased_encoder.offsets[100:150])

print("\nUncased:")
print(len(wordPiece_uncased_encoder.ids))
print(wordPiece_uncased_encoder.tokens[:20])
print(wordPiece_uncased_encoder.offsets[:20])

328241
['be', 'an', 'eye', 'opener', 'to', 'function', '##aries', 'in', 'law', 'courts', 'at', 'all', 'levels', 'that', 'delay', 'more', 'often', 'defeats', 'justice', 'in', '##var', '##iably', 'adds', 'complications', 'to', 'the', 'already', 'complicated', 'issues', 'involved', 'in', 'cases', 'coming', 'before', 'them', ',', 'and', 'makes', 'their', 'duties', 'more', 'one', '##rous', 'by', 'requiring', 'them', 'to', 'adjust', 'rights', 'and']
[(458, 460), (461, 463), (464, 467), (468, 474), (475, 477), (479, 487), (487, 492), (493, 495), (496, 499), (500, 506), (507, 509), (510, 513), (514, 520), (521, 525), (526, 531), (532, 536), (537, 542), (543, 550), (551, 558), (559, 561), (561, 564), (564, 569), (570, 574), (575, 588), (589, 591), (592, 595), (596, 603), (604, 615), (616, 622), (623, 631), (632, 634), (635, 640), (641, 647), (648, 654), (655, 659), (659, 660), (661, 664), (665, 670), (671, 676), (677, 683), (684, 688), (689, 692), (692, 696), (697, 699), (700, 709), (710, 714),

In [50]:
from transformers import BertTokenizerFast # using for research purpose only
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [52]:
train_data = df.drop(["NoOfJudges", "Decision_Status"], axis=1)
train_data.head(1)

,CaseNo,Judgement,ArgumentsOfPetitioner,ArgumentsOfResponder,Facts,Principle,Statute,CaseCited,Reasoning,Decision
0,Civil Appeal No. 6006 Of 2001 (With C.A. No. 3...,Dharmadhikari J.\t\tThese two cross appeals ha...,32. It needs to be mentioned at this stage tha...,44. The alternative argument advanced on behal...,1.These two cross appeals have been preferred ...,1.These two cross appeals have been preferred ...,"Specific Relief Act, 1963 Section 20&nbsp;",Lala Durga Prasad vs. Lala Deep Chand. (1954 S...,After hearing the argument at length advanced ...,&nbsp;70. On duly appreciating of the evidence...


In [53]:
for i in train_data:
    train_data[i] = train_data[i].apply(lambda x: tokenizer(x)["input_ids"])

Token indices sequence length is longer than the specified maximum sequence length for this model (22562 > 512). Running this sequence through the model will result in indexing errors


In [55]:
train_data.head(2)

,CaseNo,Judgement,ArgumentsOfPetitioner,ArgumentsOfResponder,Facts,Principle,Statute,CaseCited,Reasoning,Decision
0,"[101, 2942, 5574, 2053, 1012, 5174, 2575, 1997...","[101, 20669, 19114, 20224, 1046, 1012, 2122, 2...","[101, 3590, 1012, 2009, 3791, 2000, 2022, 3855...","[101, 4008, 1012, 1996, 4522, 6685, 3935, 2006...","[101, 1015, 1012, 2122, 2048, 2892, 9023, 2031...","[101, 1015, 1012, 2122, 2048, 2892, 9023, 2031...","[101, 3563, 4335, 2552, 1010, 3699, 2930, 2322...","[101, 21348, 2050, 28746, 17476, 5443, 1012, 2...","[101, 2044, 4994, 1996, 6685, 2012, 3091, 3935...","[101, 1004, 1050, 5910, 2361, 1025, 3963, 1012..."
1,"[101, 2942, 5574, 2053, 1012, 4805, 2581, 2549...","[101, 5185, 23017, 1010, 1046, 1012, 1015, 101...","[101, 1004, 1050, 5910, 2361, 1025, 4342, 9517...","[101, 1004, 1050, 5910, 2361, 1025, 4342, 9517...","[101, 1996, 10439, 24178, 2015, 9922, 9535, 29...","[101, 1004, 1050, 5910, 2361, 1025, 1996, 2691...","[101, 9740, 1997, 2833, 4639, 16754, 2552, 101...","[101, 12338, 17514, 5474, 8712, 2523, 5443, 10...","[101, 2045, 3544, 2000, 2022, 7857, 1999, 1996...","[101, 1999, 1996, 2765, 1010, 2057, 3499, 1996..."


In [58]:
# training assuming as a single textual section
df.head(1)

,CaseNo,Judgement,ArgumentsOfPetitioner,ArgumentsOfResponder,Facts,Principle,Statute,CaseCited,Reasoning,Decision,NoOfJudges,Decision_Status
0,Civil Appeal No. 6006 Of 2001 (With C.A. No. 3...,Dharmadhikari J.\t\tThese two cross appeals ha...,32. It needs to be mentioned at this stage tha...,44. The alternative argument advanced on behal...,1.These two cross appeals have been preferred ...,1.These two cross appeals have been preferred ...,"Specific Relief Act, 1963 Section 20&nbsp;",Lala Durga Prasad vs. Lala Deep Chand. (1954 S...,After hearing the argument at length advanced ...,&nbsp;70. On duly appreciating of the evidence...,2,2


In [59]:
set(df["NoOfJudges"])

{2, 3}

In [132]:
verdicts = []
counts = 0
for i in df.drop(["NoOfJudges", "Decision_Status"], axis=1).values:
    lines = ""
    for l in i:
        lines += l
    verdicts.append(lines)
print(counts)

0
